# Bangla Multimodal Fake News: Model Training

This notebook extracts features and trains the models. It needs a **T4 GPU**: Runtime > Change runtime type > T4 GPU.

Run `01_preprocessing.ipynb` first, so that `unified_split.csv` exists on Drive.

In [2]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


## 1. Unpack the project code

In [3]:
import shutil
import zipfile
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive")
CODE_DIR = Path("/content/Bangla-Multimodal-FakeNews")

candidates = list(DRIVE_ROOT.glob("Bangla-Multimodal-FakeNews*.zip")) + list(
    (DRIVE_ROOT / "BFND-data").glob("Bangla-Multimodal-FakeNews*.zip")
)
assert candidates, "Code zip not found on Drive. Upload Bangla-Multimodal-FakeNews.zip first."
ZIP = max(candidates, key=lambda p: p.stat().st_mtime)
print("Using zip:", ZIP)

shutil.rmtree(CODE_DIR, ignore_errors=True)   # remove any old copy so stale code cannot run
with zipfile.ZipFile(ZIP) as z:
    z.extractall("/content")
print("Files in zip:", len(zipfile.ZipFile(ZIP).namelist()))

Using zip: /content/drive/MyDrive/Bangla-Multimodal-FakeNews.zip
Files in zip: 74


In [4]:
import os
from pathlib import Path

os.chdir(CODE_DIR)
config_text = Path("configs/config.py").read_text(encoding="utf-8")
assert "CLIP_IMAGE_MODEL" in config_text, (
    "Old code detected. Upload the latest zip to Drive, then run the unpack cell again."
)
print("Code version OK. Working directory:", os.getcwd())

Code version OK. Working directory: /content/Bangla-Multimodal-FakeNews


## 2. Check the GPU

In [5]:
import torch

assert torch.cuda.is_available(), (
    "No GPU. Set Runtime > Change runtime type > T4 GPU, restart, and run again."
)
print("GPU:", torch.cuda.get_device_name(0))

GPU: Tesla T4


## 3. Install packages

In [6]:
!pip install -q "accelerate>=0.26" "transformers>=4.46" "sentence-transformers>=3.0" "rank-bm25>=0.2.2" joblib openpyxl

## 4. Preflight

In [7]:
!python scripts/00_preflight.py

[OK]   Hugging Face model: csebuetnlp/banglabert
[OK]   Hugging Face model: sentence-transformers/clip-ViT-B-32
[OK]   Hugging Face model: sentence-transformers/clip-ViT-B-32-multilingual-v1
[OK]   Hugging Face model: intfloat/multilingual-e5-base
[OK]   Hugging Face model: MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7
[WARN] btmd_schema.py: 'source_url_col' is not set; related features/guards will be skipped
[OK]   btmd_schema.py: required fields are filled in
[OK]   BTMD slim data: /content/drive/MyDrive/BFND-data/data/raw
[OK] evidence.csv: /content/drive/MyDrive/BFND-data/data/evidence/evidence.csv
[MISS] cultural_annotations.csv (manual): /content/drive/MyDrive/BFND-data/data/annotations/cultural_annotations.csv
[OK] cultural_silver_labels.csv (LLM): /content/drive/MyDrive/BFND-data/data/annotations/cultural_silver_labels.csv
[MISS] claim_evidence_pairs.csv: /content/drive/MyDrive/BFND-data/data/annotations/claim_evidence_pairs.csv
[OK]   GPU: Tesla T4

Experiments tha

## 5. Feature extraction

Builds BanglaBERT embeddings and CLIP image and text embeddings. The cultural and evidence features are built only when their input files exist; otherwise a [SKIP] line is printed.

In [8]:
!python scripts/02_extract_features.py

config.json: 100% 586/586 [00:00<00:00, 2.40MB/s]
tokenizer_config.json: 100% 119/119 [00:00<00:00, 431kB/s]
vocab.txt: 100% 528k/528k [00:00<00:00, 102MB/s]
special_tokens_map.json: 100% 112/112 [00:00<00:00, 528kB/s]

pytorch_model.bin: downloading bytes:  77% 339M/443M [00:02<00:00, 208MB/s, 28.9MB/s  ]
pytorch_model.bin: downloading bytes:  94% 416M/443M [00:02<00:00, 272MB/s, 33.3MB/s  ]
pytorch_model.bin: downloading bytes: 100% 419M/419M [00:03<00:00, 121MB/s, 36.8MB/s  ]
pytorch_model.bin: reconstructing file: 100% 443M/443M [00:03<00:00, 128MB/s, 40.1MB/s  ]
Loading weights: 100% 197/197 [00:00<00:00, 26761.17it/s]
[transformers] ElectraModel LOAD REPORT from: csebuetnlp/banglabert
Key                                               | Status     |  | 
--------------------------------------------------+------------+--+-
discriminator_predictions.dense.weight            | UNEXPECTED |  | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED |  | 
discriminator_prediction

## 6. Check the feature files

Expected shapes: `bert (3910, 768)`, `clip_img (3910, 512)`, `clip_txt (3910, 512)`, `sim (3910, 1)`. The row count changes if the data changes.

In [9]:
import numpy as np
import pandas as pd
from configs.config import FEATURE_DIR

meta = pd.read_csv(FEATURE_DIR / "meta.csv")
print(meta["split"].value_counts())
for name in ["bert", "clip_img", "clip_txt", "sim"]:
    print(name, np.load(FEATURE_DIR / f"{name}.npy").shape)

split
train    2736
test      588
val       586
Name: count, dtype: int64
bert (3910, 768)
clip_img (3910, 512)
clip_txt (3910, 512)
sim (3910, 1)


## 7. Length-only baseline

A model that sees only the text length. If it scores close to the real models, those models may be relying on length, so their results need careful reading.

In [10]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

from configs.config import FEATURE_DIR

meta = pd.read_csv(FEATURE_DIR / "meta.csv")
meta["n_chars"] = meta["model_text"].astype(str).str.len()
X_len = np.log1p(meta[["n_chars"]].to_numpy(dtype=float))
train_mask = meta["split"].eq("train").to_numpy()
test_mask = meta["split"].eq("test").to_numpy()
y = meta["y"].astype(int).to_numpy()

clf = LogisticRegression(max_iter=1000, class_weight="balanced")
clf.fit(X_len[train_mask], y[train_mask])
pred = clf.predict(X_len[test_mask])
print("Length-only accuracy (test):", round((pred == y[test_mask]).mean(), 4))
print("Length-only macro-F1 (test):", round(f1_score(y[test_mask], pred, average="macro"), 4))

Length-only accuracy (test): 0.483
Length-only macro-F1 (test): 0.3846


## 8. E1: fine-tune BanglaBERT (text only)

Runs three seeds. Expect a long run, roughly 20 to 40 minutes on a T4.

In [11]:
!python scripts/03_train_text_baseline.py

seed 42: already finished, loaded from disk
seed 42: accuracy=0.8061 macro_f1=0.5445
Loading weights: 100% 197/197 [00:00<00:00, 25605.14it/s]
[transformers] ElectraForSequenceClassification LOAD REPORT from: csebuetnlp/banglabert
Key                                               | Status     | 
--------------------------------------------------+------------+-
discriminator_predictions.dense_prediction.bias   | UNEXPECTED | 
discriminator_predictions.dense.weight            | UNEXPECTED | 
discriminator_predictions.dense.bias              | UNEXPECTED | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED | 
classifier.dense.bias                             | MISSING    | 
classifier.out_proj.bias                          | MISSING    | 
classifier.out_proj.weight                        | MISSING    | 
classifier.dense.weight                           | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect iden

## 9. Fusion experiments, decision layer and evidence check

In [12]:
!python scripts/04_train_fusion.py

E0_has_image_only seed=42: acc=0.2245 macroF1=0.2023
E0_has_image_only seed=43: acc=0.2245 macroF1=0.2023
E0_has_image_only seed=44: acc=0.2245 macroF1=0.2023
E1f_frozen_banglabert_mlp seed=42: acc=0.7568 macroF1=0.5880
E1f_frozen_banglabert_mlp seed=43: acc=0.7789 macroF1=0.6463
E1f_frozen_banglabert_mlp seed=44: acc=0.7874 macroF1=0.6471
E2_clip_image_only seed=42: acc=0.5952 macroF1=0.5335
E2_clip_image_only seed=43: acc=0.6071 macroF1=0.5557
E2_clip_image_only seed=44: acc=0.6003 macroF1=0.5347
E3_banglabert_clip seed=42: acc=0.8333 macroF1=0.7266
E3_banglabert_clip seed=43: acc=0.8333 macroF1=0.7426
E3_banglabert_clip seed=44: acc=0.8452 macroF1=0.7652
E5_plus_evidence seed=42: acc=0.8282 macroF1=0.7240
E5_plus_evidence seed=43: acc=0.8214 macroF1=0.7248
E5_plus_evidence seed=44: acc=0.8282 macroF1=0.7238
E6: {'min_confidence_from_validation': 0.7224954962730408, 'target_coverage': 0.85, 'final_label_counts_test': {'Insufficient': 376, 'Authentic': 138, 'Misleading': 68, 'Satire':

## 10. Summary

In [13]:
from configs.config import RESULTS_DIR

print((RESULTS_DIR / "summary.md").read_text(encoding="utf-8"))

| Experiment | Features | Accuracy (mean ± std) | Macro-F1 (mean ± std) |
|---|---|---|---|
| E1_banglabert_finetuned | text (fine-tuned) | 0.8090 ± 0.0040 | 0.5458 ± 0.0023 |
| E0_has_image_only | has_image | 0.2245 ± 0.0000 | 0.2023 ± 0.0000 |
| E1f_frozen_banglabert_mlp | bert | 0.7744 ± 0.0129 | 0.6271 ± 0.0277 |
| E2_clip_image_only | clip_img | 0.6009 ± 0.0049 | 0.5413 ± 0.0102 |
| E3_banglabert_clip | bert, clip_img, clip_txt, sim | 0.8373 ± 0.0056 | 0.7448 ± 0.0158 |
| E5_plus_evidence | bert, clip_img, clip_txt, sim, evidence | 0.8260 ± 0.0032 | 0.7242 ± 0.0004 |

